# Finalize: the training set
Input: `dataset_final/`, the sentence chunks from step 5 of `05_remove_fillers`. It is not changed.
Output: **`dataset_train/`**, the set to train on: `chunks/*.wav`, `metadata.json`, and the manifests `train.txt` and `val.txt`.
Both folders are in the run folder that `RUN` in `pipeline_paths.py` sets.

**Kernel: `.venv-nemo`.** It has NeMo for the speaker model, and MMS runs there too. Models:
- MMS-1b-all with the Uzbek (Latin) adapter, as in 05;
- TitaNet-large, for voice embeddings (`nvidia/speakerverification_en_titanet_large`).

Run the cells from top to bottom. Steps 1–3 measure; what they compute is saved in `outputs/train/`. After changing a setting of steps 4–5, only those two cells need to run again, and they take a few minutes.

**1. Diarization check:** what every diarization system of 01 hears inside each final chunk.
- The diarizations (`outputs/diarization/<name>/<system>/`) are followed through every cut made since 01: the interruptions cut out in 02, the fillers cut in 05 and the sentence cuts of step 5. This uses `outputs/chunks/*/chunks.json` from 02.
- The **reference** is the diarization the chunks were cut by (`DIARIZATION` in `pipeline_paths.py`: `nemotron` by default). It gives each chunk its speaker label, and its **overlap**: how long two people talk at once.
- Every other system votes, unless it hears almost nothing in the chunk:
  - **agree:** it hears the chunk as the same speaker it hears the rest of that label's speech as. A 4-speaker model that merged two people still agrees;
  - **mixed:** it hears another speaker alone for at least `MIN_OTHER_SEC` inside the chunk.
- If those files are missing, the check is skipped with a warning, and nothing is dropped for it.

**2. Text:** numbers and symbols are written as words, and each text is checked against its audio.
- Every number gets its possible readings, for example:
  - 2024 → "ikki ming yigirma to'rt";
  - 1990-yil → "bir ming to'qqiz yuz to'qsoninchi yil";
  - 5-sinf → "beshinchi sinf";
  - 5ta → "beshta";
  - 2,5 → "ikki butun besh" or "ikki yarim";
  - 10-15 → "o'n o'n besh" or "o'ndan o'n beshgacha";
  - 5% → "besh foiz";
  - $100 → "yuz dollar".
- MMS listens to the chunk and keeps the reading that fits the audio best, so the text says what the speaker said.
- **align_prob** is the mean letter probability of the forced alignment of the final text. A low value means the text does not match the audio: a transcription error, or a number said in a way none of the readings has.
- `text` is the new text; `text_original` keeps the old one where it changed.

**3. Speakers:** a TitaNet voice embedding for every chunk.
- Each speaker label from the diarization (one set per podcast) gets an average voice, made from its chunks of at least `MIN_EMB_SEC`.
- **Same person:** labels whose average voices are at least `MERGE_SIM` alike become one speaker. This works across podcasts (a host in several episodes) and inside one (a speaker the diarization split in two).
- Per chunk: the other speaker of its podcast its voice is closest to, and how much closer than its own label. Step 4 uses that.

**4. Decide.** A chunk is fixed where it can be, and dropped only where it can't.

**Doubted chunks:** most systems (`MIN_VOTES`) hear the chunk as someone other than its label says. Then its voice decides: TitaNet on the denoised chunk, compared with each speaker's average voice, made from chunks the systems agree on.
- **Kept:** the voice is still closest to its own label.
- **Relabeled** (kept): the voice is at least `RELABEL_MARGIN` closer to another speaker of the podcast. It gets that speaker's label.
- **Dropped** as `uncertain_speaker`: the voice is too close to call, or the chunk is under `MIN_EMB_SEC` and the systems don't agree on another speaker.

Chunks the systems agree on keep their label, even if their voice leans elsewhere. They are listed on the review page.

**Dropped:**
- characters a TTS can't read (digits left over, Cyrillic, other symbols);
- `align_prob` below `MIN_ALIGN_PROB`: the text does not match the audio;
- an overlap of at least `MAX_OVERLAP_SEC`;
- **mixed speakers:** most systems hear another speaker alone inside the chunk;
- **uncertain speaker:** see above;
- speakers with less than `MIN_SPEAKER_MIN` minutes in total (0, the default, keeps everyone).

Dropped chunks are linked into `outputs/train/dropped/<reason>/`. The review page, `outputs/train/review/index.html`, has players for merged speakers, near misses, and relabeled, mixed and wrong-speaker chunks.

**5. Build** `dataset_train/`:
- every chunk is brought to `TARGET_LUFS` loudness (ITU-R BS.1770), never above `PEAK_DBFS`, and stays at 48 kHz (`OUT_SR`, the rate of the sources: nothing is resampled);
- `metadata.json` has `sample_rate` and `speakers` (each new id with its old labels and minutes). Per chunk it has `file, speaker, text, duration, split`, plus `speaker_label` (`speaker_label_before` when relabeled), `diarization` (systems that agree / that voted), `text_original`, `align_prob`, `overlap_sec` and `gain_db`;
- `train.txt` and `val.txt` have one line per chunk, `chunks/<name>.wav|text|speaker`. This is the LJSpeech/VITS style most trainers read;
- `VAL_SHARE` of each speaker's chunks go to validation.

In [1]:
import html
import importlib
import json
import logging
import math
import os
import random
import re
import shutil
import time
import warnings
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pyloudnorm
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
import pipeline_paths as P
importlib.reload(P)           # a change of RUN in pipeline_paths.py counts without restarting the kernel
print(P.describe())

run 'test': reading and writing in /home/intern/projects/tts/runs/test


In [2]:
SRC = P.DATASET_FINAL         # input: the sentence chunks of 05 step 5, not changed
OUT = P.DATASET_TRAIN         # output, rebuilt on every run of step 5
WORK = P.OUTPUTS / "train"    # text.json, speakers.npz, review/, report.json, dropped/

# 1. diarization check
MIN_OTHER_SEC = 0.5           # a system that hears another speaker alone this long inside a chunk votes "mixed"
MIN_VOTES = None              # votes that make a chunk mixed or its label wrong; None = a majority of the systems

# 3. speakers
MIN_EMB_SEC = 2.0             # shorter chunks don't count for a speaker's voice and are not judged (too little audio)
MERGE_SIM = 0.70              # speaker labels at least this alike (cosine of their average voices) become one speaker

# 4. decide
MIN_ALIGN_PROB = 0.40         # drop texts that match their audio worse than this (mean letter probability)
MAX_OVERLAP_SEC = 0.5         # drop chunks with this much of two people talking at once
RELABEL = True                # a doubted chunk whose voice is clearly another speaker's gets that label (False: dropped)
RELABEL_MARGIN = 0.05         # ...at least this much more alike (cosine) to that speaker's voice than to its own label's
WRONG_MARGIN = 0.10           # review only: chunks the systems agree on whose voice is this much closer to another
MIN_SPEAKER_MIN = 0.0         # drop speakers with less than this many minutes left (0 = keep every speaker)

# 5. build
OUT_SR = P.SAMPLE_RATE        # 48 kHz: dataset_train/ keeps the rate of the sources, nothing is resampled
TARGET_LUFS = -23.0           # loudness of every chunk
PEAK_DBFS = -1.0              # ...but no sample above this
VAL_SHARE = 0.02              # share of each speaker's chunks for validation (speakers with at least 1 / VAL_SHARE chunks)
SEED = 0

WORKERS = 8
N_REVIEW = 8
SAVE_EVERY = 200

## Data and functions

In [3]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
src, out_dir, work = Path(SRC), Path(OUT), Path(WORK)
work.mkdir(parents=True, exist_ok=True)
meta = json.load(open(src / "metadata.json", encoding="utf-8"))
SR = meta["sample_rate"]
chunks = meta["chunks"]
by_file = {c["file"]: c for c in chunks}
logging.getLogger("nemo_logger").setLevel(logging.ERROR)


def save_json(path, obj, indent=1):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    os.replace(tmp, path)


def read16(path):
    x, sr = sf.read(path, dtype="float32")
    return AF.resample(torch.from_numpy(x), sr, 16000).numpy()


def link(a, b):
    if b.exists():
        return                        # linked before (the same sample twice on a review page)
    b.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(a, b)                 # a hard link: no extra space
    except OSError:
        shutil.copy2(a, b)


print(f"{src}: {len(chunks)} chunks, {sum(c['duration'] for c in chunks) / 3600:.2f} h, {SR} Hz, "
      f"{len({c['speaker'] for c in chunks})} speaker labels; device {dev}")

/home/intern/projects/tts/runs/test/dataset_final: 604 chunks, 1.19 h, 48000 Hz, 9 speaker labels; device cuda


## 1. Diarization check

In [4]:
JOIN_FADE_SEC = 0.02          # 02 joined the parts of a chunk with this crossfade
PAUSE_SEC = (0.10, 0.35)      # 05 put room tone where it cut inside a chunk: 40% of the cut, within these limits
RES = 100                     # 10 ms frames
REF = P.DIARIZATION           # the diarization the chunks were cut by

nf = {c["file"]: c for c in json.load(open(P.DATASET_NOFILLERS / "metadata.json", encoding="utf-8"))["chunks"]}
clean_meta = P.DATASET_CLEAN / "metadata.all.json"     # 04 keeps the full list here
clean_meta = clean_meta if clean_meta.exists() else P.DATASET_CLEAN / "metadata.json"
clean_dur = {c["file"]: c["duration"] for c in json.load(open(clean_meta, encoding="utf-8"))["chunks"]}


def read_rttm(path):
    turns = []
    for line in path.read_text().splitlines():
        p = line.split()
        s = float(p[-7])
        turns.append((s, s + float(p[-6]), p[-3]))
    return turns


def activity(turns, n):
    # labels, and a (labels x frames) matrix: who is talking in every 10 ms frame
    labels = sorted({k for *_, k in turns})
    A = np.zeros((len(labels), n), bool)
    for s, e, k in turns:
        A[labels.index(k), max(0, round(s * RES)):min(n, round(e * RES))] = True
    return labels, A


def splice_frames(m, cuts, dur):
    # the frames (last axis) of a chunk after 05 cut it, the same way 05's splice() cut the audio
    regions = []
    for s, e in sorted((max(0.0, s), min(dur, e)) for s, e, *_ in cuts):
        if regions and s <= regions[-1][1]:
            regions[-1][1] = max(regions[-1][1], e)
        elif e > s:
            regions.append([s, e])
    out, pos = [], 0
    for s, e in regions:
        a, b = round(s * RES), round(e * RES)
        out.append(m[..., pos:a])
        if int(s * SR) > 0 and int(e * SR) < round(dur * SR):
            out.append(np.zeros(m.shape[:-1] + (round(float(np.clip(0.4 * (e - s), *PAUSE_SEC)) * RES),), bool))
        pos = b
    out.append(m[..., pos:])
    return np.concatenate(out, axis=-1)


frames = {}         # dataset_nofillers file -> {system: (labels x frames) on the chunk's own clock}
system_labels = {}  # (podcast, system) -> the system's labels
main = {}           # (podcast, system) -> {reference label: the system's label most of that speaker's speech gets}
systems, missing = set(), []
for num, name in meta["podcasts"].items():
    base, cj = P.OUTPUTS / "diarization" / name, P.OUTPUTS / "chunks" / name / "chunks.json"
    found = sorted(p.parent.name for p in base.glob(f"*/{name}.rttm")) if base.exists() else []
    if not cj.exists() or REF not in found:
        missing.append(name)
        continue
    rec = json.load(open(cj, encoding="utf-8"))
    n = math.ceil(rec["duration"] * RES) + 1
    acts = {s: activity(read_rttm(base / s / f"{name}.rttm"), n) for s in found}
    systems |= set(found)
    ref_labels, R = acts[REF]
    for s, (labels, A) in acts.items():
        system_labels[num, s] = labels
        co = np.array([[int((A[i] & R[j]).sum()) for j in range(len(ref_labels))] for i in range(len(labels))])
        main[num, s] = {r: labels[int(co[:, j].argmax())] for j, r in enumerate(ref_labels)} if len(labels) else {}
    for ch in rec["chunks"]:
        f = f"chunks/{num}_{Path(ch['file']).name}"
        if f not in nf:
            continue                                          # left out on the way
        cuts = (nf[f].get("fillers") or {}).get("cuts")
        frames[f] = {}
        for s, (labels, A) in acts.items():
            m = np.concatenate([A[:, round(a * RES):round(b * RES)][:, round(JOIN_FADE_SEC * RES) if k else 0:]
                                for k, (a, b) in enumerate(ch["parts"])], axis=1)
            frames[f][s] = splice_frames(m, cuts, clean_dur.get(f, m.shape[1] / RES)) if cuts else m

voters = sorted(systems - {REF, "consensus"})
need = MIN_VOTES or len(voters) // 2 + 1                   # a majority of the systems


def check(c):
    # what every system hears in the final chunk c; None when its diarization is missing
    num, own = c["speaker"][:3], c["speaker"][4:]          # "001_speaker_0" -> "001", "speaker_0"
    if any(f not in frames for f, *_ in c["sources"]):
        return None
    per = defaultdict(list)
    for f, a, b in c["sources"]:
        k = frames[f][REF].shape[1] / RES / nf[f]["duration"]   # 1.0 unless the mapping is off
        for s, m in frames[f].items():
            per[s].append(m[:, round(a * RES * k):round(b * RES * k)])
    out = dict(votes=0, agree=0, mixed=0, dom={}, other_sec={}, overlap_sec=0.0)
    for s, parts in per.items():
        m = np.concatenate(parts, axis=1)
        if s == REF:
            out["overlap_sec"] = round(float((m.sum(0) >= 2).sum()) / RES, 2)
            continue
        if s not in voters or not m.shape[0] or m.any(0).sum() < 0.3 * RES:
            continue                                        # the system hears (almost) nothing here: no vote
        dom = int(m.sum(1).argmax())
        other = float(((m.sum(0) == 1) & ~m[dom]).sum()) / RES
        out["votes"] += 1
        out["dom"][s] = system_labels[num, s][dom]
        out["other_sec"][s] = round(other, 2)
        out["agree"] += int(system_labels[num, s][dom] == main[num, s].get(own))
        out["mixed"] += int(other >= MIN_OTHER_SEC)
    return out


diar = {c["file"]: check(c) for c in chunks}
known = [d for d in diar.values() if d]
if missing:
    print(f"WARNING: no outputs/chunks/<name>/chunks.json or no {REF} RTTM for {len(missing)} podcasts; their chunks "
          f"are not checked: {missing[:5]}")
print(f"reference: {REF}; voting systems: {voters} (a majority: {need}); chunks checked: {len(known)} of {len(chunks)}")
if known:
    print("systems that agree with the chunk's label (agree/voted):",
          dict(sorted(Counter(f"{d['agree']}/{d['votes']}" for d in known).items())))
    print(f"most systems hear someone else: {sum(d['votes'] - d['agree'] >= need for d in known)} chunks; "
          f"most hear another speaker alone for >= {MIN_OTHER_SEC} s (mixed): {sum(d['mixed'] >= need for d in known)}; "
          f"overlap >= {MAX_OVERLAP_SEC} s: {sum(d['overlap_sec'] >= MAX_OVERLAP_SEC for d in known)}")

reference: nemotron; voting systems: ['clustering', 'sortformer_v2', 'sortformer_v2_1'] (a majority: 2); chunks checked: 604 of 604
systems that agree with the chunk's label (agree/voted): {'0/1': 5, '0/2': 18, '0/3': 40, '1/1': 7, '1/2': 22, '1/3': 125, '2/2': 23, '2/3': 121, '3/3': 243}
most systems hear someone else: 183 chunks; most hear another speaker alone for >= 0.5 s (mixed): 10; overlap >= 0.5 s: 3


## 2. Text: numbers as words, and how well the text matches the audio
Only chunks whose text has a number or a symbol need MMS to choose a reading. If `dataset_final/metadata.json` has no `align_prob` yet, every chunk is aligned once (about 2–3 min per hour of audio); that is saved in `text.json`.

In [5]:
ONES = ["nol", "bir", "ikki", "uch", "to'rt", "besh", "olti", "yetti", "sakkiz", "to'qqiz"]
TENS = ["", "o'n", "yigirma", "o'ttiz", "qirq", "ellik", "oltmish", "yetmish", "sakson", "to'qson"]
DEN = {1: "o'ndan", 2: "yuzdan", 3: "mingdan"}
CUR = {"$": "dollar", "€": "yevro", "₽": "rubl", "£": "funt", "%": "foiz"}
SYM = {"+": ["plyus"], "=": ["teng"], "&": ["va", "end"], "№": ["raqam", "nomer"], "°": ["daraja", "gradus"],
       "%": ["foiz", "protsent"]}


def cardinal(n, one=False):
    # 2024 -> "ikki ming yigirma to'rt"; one=True says "bir yuz", "bir ming" instead of "yuz", "ming"
    if n < 10:
        return ONES[n]
    if n < 100:
        return " ".join(w for w in (TENS[n // 10], ONES[n % 10] if n % 10 else "") if w)
    if n < 1000:
        head = ("bir yuz" if one else "yuz") if n // 100 == 1 else f"{ONES[n // 100]} yuz"
        return head + (f" {cardinal(n % 100)}" if n % 100 else "")
    for size, name in [(10**9, "milliard"), (10**6, "million"), (1000, "ming")]:
        if n >= size:
            q = n // size
            head = (f"bir {name}" if one else name) if (q == 1 and size == 1000) else f"{cardinal(q, one)} {name}"
            return head + (f" {cardinal(n % size, one)}" if n % size else "")


def ordinal(words):
    # "ikki" -> "ikkinchi", "bir ming to'qqiz yuz to'qson" -> "... to'qsoninchi"
    return words + ("nchi" if words[-1] in "aeiou" else "inchi")


def attach(words, suffix):
    # "besh" + "ta" -> "beshta"; "ellik" + "ga" -> "ellikka"; "qirq" + "gacha" -> "qirqqacha"
    if suffix[:1].lower() == "g" and words[-1] in "kq":
        suffix = words[-1] + suffix[1:]
    return words + suffix


def numbers(s):
    # the readings of a number written as digits, maybe in groups ("10 000")
    groups = s.split()
    digits = "".join(groups)
    if len(digits) > 12 or (len(digits) > 1 and digits[0] == "0"):
        return [" ".join(ONES[int(d)] for d in digits)]       # a code or a phone number: digit by digit
    v = int(digits)
    out = [cardinal(v), cardinal(v, one=True)]
    if len(groups) > 1 and not any(g[0] == "0" for g in groups[1:]):
        out.append(" ".join(cardinal(int(g)) for g in groups))   # "2 100": maybe two numbers
    return list(dict.fromkeys(out))


def decimals(n, sep, d):
    out = []
    for r in numbers(n):
        out += [f"{r} butun {cardinal(int(d))}", f"{r} {cardinal(int(d))}"]
        if len(d) in DEN:
            out.append(f"{r} butun {DEN[len(d)]} {cardinal(int(d))}")
        if d == "5":
            out.append(f"{r} yarim")
        out.append(f"{r} {'nuqta' if sep == '.' else 'vergul'} {cardinal(int(d))}")
    return out


W = r"[^\W\d_]+(?:['ʻʼ‘’][^\W\d_]*)*"                         # a word, maybe with apostrophes (o'n, yil)
N = r"\d{1,3}(?:[  ]\d{3})+|\d+"                         # 10 000, or plain digits
EXPR = re.compile(
    rf"(?P<cur>[$€₽£])\s?(?P<cn>{N})(?:(?P<cs>[.,])(?P<cd>\d+))?"
    rf"|(?P<pn>{N})(?:(?P<ps>[.,])(?P<pd>\d+))?\s?(?P<pc>[$€₽£%])"
    rf"|(?P<h>\d{{1,2}}):(?P<m>\d\d)\b"
    rf"|(?P<ra>\d+)\s?[-–—]\s?(?P<rb>\d+)(?:-(?P<rw>{W}))?"
    rf"|(?P<on>\d+)-(?P<ow>{W})"
    rf"|(?P<dn>{N})(?P<ds>[.,])(?P<dd>\d+)"
    rf"|(?P<gn>{N})(?P<gw>{W})"
    rf"|(?P<n>{N})"
    rf"|(?P<sym>[+=&№°%])")


def readings(m):
    # every way the matched number or symbol may have been said
    g = m.groupdict()
    if g["cur"]:
        rs = decimals(g["cn"], g["cs"], g["cd"]) if g["cs"] else numbers(g["cn"])
        return [f"{r} {CUR[g['cur']]}" for r in rs]
    if g["pc"]:
        rs = decimals(g["pn"], g["ps"], g["pd"]) if g["ps"] else numbers(g["pn"])
        words = SYM["%"] if g["pc"] == "%" else [CUR[g["pc"]]]
        return [f"{r} {w}" for r in rs for w in words]
    if g["h"]:
        h, mm = int(g["h"]), int(g["m"])
        return [f"{cardinal(h)} {cardinal(mm)}"] if mm else [cardinal(h), f"{cardinal(h)} nol nol"]
    if g["ra"]:
        a, b, w = int(g["ra"]), int(g["rb"]), g["rw"]
        if w:
            return [f"{ordinal(cardinal(a))} {ordinal(cardinal(b))} {w}", f"{cardinal(a)} {ordinal(cardinal(b))} {w}",
                    f"{cardinal(a)} {cardinal(b)} {w}"]
        return [f"{cardinal(a)} {cardinal(b)}", f"{attach(cardinal(a), 'dan')} {attach(cardinal(b), 'gacha')}",
                f"{cardinal(a)} {attach(cardinal(b), 'gacha')}"]
    if g["on"]:
        w = g["ow"]
        mark = re.fullmatch(r"(?:i?n)?chi((?:si)?(?:ning|ni|ga|da|dan|lar)?)", w, re.I)   # 2-chi, 5-inchi, 3-chisi
        if mark:
            return [ordinal(r) + mark.group(1) for r in numbers(g["on"])]
        return [x for r in numbers(g["on"]) for x in (f"{ordinal(r)} {w}", f"{r} {w}", attach(r, w))]
    if g["dd"]:
        return decimals(g["dn"], g["ds"], g["dd"])
    if g["gw"]:
        return [x for r in numbers(g["gn"]) for x in (attach(r, g["gw"]), f"{ordinal(r)} {g['gw']}", f"{r} {g['gw']}")]
    if g["n"]:
        return numbers(g["n"])
    return SYM[g["sym"]]


def render(text, ms, cands, pick):
    # the text with every match replaced by its picked reading; a sentence start gets a capital letter
    out, pos = [], 0
    for m, cs, k in zip(ms, cands, pick):
        rep, before = cs[k], text[:m.start()].rstrip()
        if not before or before[-1] in ".!?…":
            rep = rep[:1].upper() + rep[1:]
        out += [text[pos:m.start()], rep]
        pos = m.end()
    out.append(text[pos:])
    return re.sub(r"\s+", " ", "".join(out)).strip()


APOS = str.maketrans({"‘": "'", "’": "'", "ʻ": "'", "ʼ": "'", "`": "'"})
mms = None


def load_mms():
    global mms, processor, VOCAB, BLANK, LETTERS
    if mms is None:
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        processor = AutoProcessor.from_pretrained("facebook/mms-1b-all", target_lang="uzb-script_latin")
        mms = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", target_lang="uzb-script_latin",
                                             ignore_mismatched_sizes=True).eval().to(dev)
        VOCAB = processor.tokenizer.get_vocab()
        BLANK = processor.tokenizer.pad_token_id
        LETTERS = set("abcdefghijklmnopqrstuvwxyz'") & set(VOCAB)


@torch.inference_mode()
def emissions(x16):
    iv = processor.feature_extractor(x16, sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    return torch.log_softmax(mms(iv).logits.float(), -1)[0].cpu()


def ctc(em, text):
    # forced alignment of the letters of text; None when it can't be aligned (no letters, or too many for the audio)
    t = [VOCAB[ch] for ch in text.lower().translate(APOS) if ch in LETTERS]
    if not t:
        return None
    try:
        return AF.forced_align(em[None], torch.tensor([t], dtype=torch.int32), blank=BLANK)
    except Exception:
        return None


def normalize(c):
    rec = dict(text_in=c["text"], duration=c["duration"])
    ms = list(EXPR.finditer(c["text"]))
    if not ms and c.get("align_prob") is not None:
        return dict(rec, text=c["text"], align_prob=c["align_prob"])
    em = emissions(read16(src / c["file"]))
    cands = [list(dict.fromkeys(readings(m))) for m in ms]
    pick = [0] * len(ms)
    for i, cs in enumerate(cands):                            # each number in turn: the reading that fits the audio best
        if len(cs) > 1:
            def loglik(k):
                r = ctc(em, render(c["text"], ms, cands, pick[:i] + [k] + pick[i + 1:]))
                return -math.inf if r is None else float(r[1].sum())
            pick[i] = max(range(len(cs)), key=loglik)
    text = render(c["text"], ms, cands, pick)
    r = ctc(em, text)
    prob = None if r is None else round(float(np.mean([s.score for s in AF.merge_tokens(r[0][0], r[1][0].exp())])), 3)
    return dict(rec, text=text, align_prob=prob, numbers=[[m.group(), cands[i][pick[i]], len(cands[i])]
                                                          for i, m in enumerate(ms)])


text_path = work / "text.json"
texts = json.load(open(text_path, encoding="utf-8")) if text_path.exists() else {}
todo = [c for c in chunks if (texts.get(c["file"]) or {}).get("text_in") != c["text"]
        or texts[c["file"]].get("duration") != c["duration"]]
need_mms = [c for c in todo if EXPR.search(c["text"]) or c.get("align_prob") is None]
print(f"{len(chunks) - len(todo)} chunks done before; {len(todo)} to go, {len(need_mms)} of them need MMS "
      f"({sum(c['duration'] for c in need_mms) / 3600:.2f} h of audio)")
if need_mms:
    load_mms()
t0 = time.time()
try:
    for k, c in enumerate(tqdm(todo, desc="text"), 1):
        texts[c["file"]] = normalize(c)
        if k % SAVE_EVERY == 0:
            save_json(text_path, texts)
finally:
    save_json(text_path, texts)
changed = [f for f in by_file if texts[f]["text"] != texts[f]["text_in"]]
probs = np.array([texts[f]["align_prob"] for f in by_file if texts[f]["align_prob"] is not None])
print(f"done in {(time.time() - t0) / 60:.1f} min; texts with numbers or symbols written as words: {len(changed)}")
print("align_prob percentiles 1/5/10/25/50:", np.round(np.percentile(probs, [1, 5, 10, 25, 50]), 3),
      f"| below MIN_ALIGN_PROB ({MIN_ALIGN_PROB}): {(probs < MIN_ALIGN_PROB).sum()}")
for (orig, chosen), n in Counter((o, ch) for f in changed for o, ch, _ in texts[f].get("numbers", [])).most_common(15):
    print(f"  {orig!r:16} -> {chosen}" + (f"  ({n} times)" if n > 1 else ""))

0 chunks done before; 604 to go, 604 of them need MMS (1.19 h of audio)
done in 2.8 min; texts with numbers or symbols written as words: 31
align_prob percentiles 1/5/10/25/50: [0.215 0.396 0.476 0.634 0.793] | below MIN_ALIGN_PROB (0.4): 31
  '100'            -> yuz  (7 times)
  '40'             -> qirq  (7 times)
  '10'             -> o'n  (3 times)
  '100%'           -> yuz foiz  (2 times)
  '24'             -> yigirma to'rt  (2 times)
  '24ga'           -> yigirma to'rtga  (2 times)
  '16'             -> o'n olti
  '12da'           -> o'n ikkida
  '15'             -> o'n besh
  '45gacha'        -> qirq beshgacha
  '20'             -> yigirma
  '1'              -> bir
  '10%'            -> o'n foiz
  '15%'            -> o'n besh foiz
  '10-15'          -> o'n o'n besh


## 3. Speakers

In [6]:
emb_path = work / "speakers.npz"
have = {}
if emb_path.exists():
    z = np.load(emb_path)
    have = {str(f): (float(d), e) for f, d, e in zip(z["files"], z["durations"], z["embs"])}
todo = [c for c in chunks if c["file"] not in have or abs(have[c["file"]][0] - c["duration"]) > 1e-3]
print(f"{len(chunks) - len(todo)} chunks embedded before; {len(todo)} to go "
      f"({sum(c['duration'] for c in todo) / 3600:.2f} h of audio)")
if todo:
    from nemo.collections.asr.models import EncDecSpeakerLabelModel
    titanet = EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large").eval().to(dev)
    todo.sort(key=lambda c: c["duration"])
    batches, cur = [], []
    for c in todo:                                            # batches of similar length, up to 300 s of audio
        if cur and (len(cur) == 64 or sum(x["duration"] for x in cur) + c["duration"] > 300):
            batches.append(cur)
            cur = []
        cur.append(c)
    batches.append(cur)
    with torch.inference_mode():
        for batch in tqdm(batches, desc="embed"):
            xs = [read16(src / c["file"]) for c in batch]
            lens = torch.tensor([len(x) for x in xs], device=dev)
            sig = torch.zeros(len(xs), int(lens.max()), device=dev)
            for i, x in enumerate(xs):
                sig[i, :len(x)] = torch.from_numpy(x)
            _, emb = titanet.forward(input_signal=sig, input_signal_length=lens)
            emb = torch.nn.functional.normalize(emb.float(), dim=-1).cpu().numpy()
            for c, e in zip(batch, emb):
                have[c["file"]] = (c["duration"], e)
    tmp = emb_path.with_name("speakers.tmp.npz")
    files = sorted(have)
    np.savez(tmp, files=np.array(files), durations=np.array([have[f][0] for f in files]),
             embs=np.stack([have[f][1] for f in files]))
    os.replace(tmp, emb_path)

E = {c["file"]: have[c["file"]][1] for c in chunks}
by_label = defaultdict(list)
for c in chunks:
    by_label[c["speaker"]].append(c)
labels = sorted(by_label)


def confirmed(c):
    # the diarization systems agree with the chunk's label (or there is no check for it)
    d = diar.get(c["file"])
    return d is None or d["votes"] - d["agree"] < need


def centroid(cs):
    # a label's voice: the duration-weighted mean embedding of its confirmed chunks of at least MIN_EMB_SEC
    cs = ([c for c in cs if c["duration"] >= MIN_EMB_SEC and confirmed(c)] or
          [c for c in cs if c["duration"] >= MIN_EMB_SEC] or cs)
    v = sum(E[c["file"]] * c["duration"] for c in cs)
    return v / np.linalg.norm(v)


cent = {l: centroid(by_label[l]) for l in labels}


def sim(f, l):
    return float(E[f] @ cent[l])


sim_own = {c["file"]: sim(c["file"], c["speaker"]) for c in chunks}
closer = {}                   # file -> (the other label of its podcast its voice is clearly closer to, similarity)
for c in chunks:
    others = [l for l in labels if l[:3] == c["speaker"][:3] and l != c["speaker"]]
    if c["duration"] >= MIN_EMB_SEC and others:
        o = max(others, key=lambda l: sim(c["file"], l))
        if sim(c["file"], o) >= sim_own[c["file"]] + WRONG_MARGIN:
            closer[c["file"]] = (o, round(sim(c["file"], o), 3))

# same person: average-linkage clustering of the labels' average voices
C = np.stack([cent[l] for l in labels])
S = C @ C.T
if len(labels) > 1:
    from scipy.cluster.hierarchy import fcluster, linkage
    cluster = fcluster(linkage(np.clip(1 - S, 0, 2)[np.triu_indices(len(labels), 1)], "average"),
                       t=1 - MERGE_SIM, criterion="distance")
else:
    cluster = np.array([1])
minutes = {l: sum(c["duration"] for c in by_label[l]) / 60 for l in labels}
groups = defaultdict(list)
for l, k in zip(labels, cluster):
    groups[k].append(l)
order = sorted(groups.values(), key=lambda ls: -sum(minutes[l] for l in ls))
new_id = {l: f"spk{i:03d}" for i, ls in enumerate(order) for l in ls}
speakers = {f"spk{i:03d}": dict(labels=ls, minutes=round(sum(minutes[l] for l in ls), 1)) for i, ls in enumerate(order)}

print(f"{len(labels)} speaker labels -> {len(speakers)} speakers; chunks whose voice is at least {WRONG_MARGIN} closer to "
      f"another speaker of their podcast: {len(closer)}")
print(f"{'speaker':8} {'minutes':>8}  labels (minutes; similarity of the voice to the other labels)")
for sid, s in speakers.items():
    parts = [f"{l} ({minutes[l]:.1f}" + (f"; {max(S[labels.index(l), labels.index(o)] for o in s['labels'] if o != l):.2f}"
                                          if len(s["labels"]) > 1 else "") + ")" for l in s["labels"]]
    print(f"{sid:8} {s['minutes']:8.1f}  {', '.join(parts)}")
near = sorted(((S[i, j], labels[i], labels[j]) for i in range(len(labels)) for j in range(i + 1, len(labels))
               if new_id[labels[i]] != new_id[labels[j]] and S[i, j] >= MERGE_SIM - 0.15), reverse=True)
if near:
    print("not merged, but close:", ", ".join(f"{a} ~ {b} {s:.2f}" for s, a, b in near[:8]))

0 chunks embedded before; 604 to go (1.19 h of audio)
[NeMo I 2026-11-27 16:20:38 save_restore_connector:309] Model EncDecSpeakerLabelModel was successfully restored from /home/intern/.cache/huggingface/hub/models--nvidia--speakerverification_en_titanet_large/snapshots/0dc382f40121a5fbd34db10a2bb04d826c2be6a8/speakerverification_en_titanet_large.nemo.
9 speaker labels -> 9 speakers; chunks whose voice is at least 0.1 closer to another speaker of their podcast: 38
speaker   minutes  labels (minutes; similarity of the voice to the other labels)
spk000       42.1  001_speaker_0 (42.1)
spk001        7.6  002_speaker_1 (7.6)
spk002        7.3  002_speaker_6 (7.3)
spk003        6.9  002_speaker_2 (6.9)
spk004        2.7  002_speaker_3 (2.7)
spk005        1.8  002_speaker_5 (1.8)
spk006        1.8  002_speaker_7 (1.8)
spk007        1.3  002_speaker_4 (1.3)
spk008        0.3  002_speaker_0 (0.3)


## 4. Decide

In [7]:
UNREADABLE = re.compile(r"[^A-Za-z'‘’ʻʼ`\s.,!?…:;\"«»“”()\[\]\-–—]")   # what is left that a TTS can't read
decision, label_of, relabeled = {}, {}, {}     # label_of: the speaker label each chunk ends up with
doubted_kept = set()                           # doubted by the systems, but the voice is still its label's
for c in chunks:
    f, t, d = c["file"], texts[c["file"]], diar.get(c["file"])
    label_of[f] = c["speaker"]
    num = c["speaker"][:3]
    if UNREADABLE.search(t["text"]):
        decision[f] = "unreadable"
    elif t["align_prob"] is None or t["align_prob"] < MIN_ALIGN_PROB:
        decision[f] = "bad_alignment"
    elif d and d["overlap_sec"] >= MAX_OVERLAP_SEC:
        decision[f] = "overlap"
    elif d and d["mixed"] >= need:
        decision[f] = "mixed_speakers"
    elif d and d["votes"] - d["agree"] >= need:              # most systems doubt the label: the voice decides
        cands = [l for l in labels if l[:3] == num]
        if c["duration"] >= MIN_EMB_SEC:
            best = max(cands, key=lambda l: sim(f, l))
            new = None if best == c["speaker"] else best if sim(f, best) >= sim_own[f] + RELABEL_MARGIN else "?"
        else:                                                # too short for the voice: the systems must agree on who
            support = {l: sum(dl == main[num, s].get(l[4:]) for s, dl in d["dom"].items()) for l in cands
                       if l != c["speaker"]}
            best = max(support, key=support.get, default=None)
            new = best if best and support[best] >= need else "?"
        if new is None:
            decision[f] = "keep"                             # the voice is still its own label's
            doubted_kept.add(f)
        elif new != "?" and RELABEL:
            label_of[f], relabeled[f] = new, c["speaker"]
            decision[f] = "keep"
        else:
            decision[f] = "uncertain_speaker"
    else:
        decision[f] = "keep"
left = Counter()
for c in chunks:
    if decision[c["file"]] == "keep":
        left[new_id[label_of[c["file"]]]] += c["duration"] / 60
small = {s for s in speakers if left[s] < MIN_SPEAKER_MIN}
for c in chunks:
    if decision[c["file"]] == "keep" and new_id[label_of[c["file"]]] in small:
        decision[c["file"]] = "small_speaker"

hours = lambda fs: sum(by_file[f]["duration"] for f in fs) / 3600
print(f"{'':17} {'chunks':>7} {'hours':>7}")
for r in ["keep", "unreadable", "bad_alignment", "overlap", "mixed_speakers", "uncertain_speaker", "small_speaker"]:
    fs = [f for f, d in decision.items() if d == r]
    print(f"{r:17} {len(fs):7d} {hours(fs):7.2f}" + (f"   (of them relabeled: {sum(1 for f in fs if f in relabeled)}; "
                                                     f"doubted by the systems but the voice confirms the label: "
                                                     f"{len(doubted_kept)})" if r == "keep" else ""))
if relabeled:
    print("relabeled:", dict(Counter(f"{relabeled[f]} -> {label_of[f]}" for f in relabeled if decision[f] == "keep")))
if small:
    print("speakers left out (less than MIN_SPEAKER_MIN minutes):", {s: round(left[s], 1) for s in sorted(small)})
odd = Counter(ch for f, d in decision.items() if d == "unreadable" for ch in UNREADABLE.findall(texts[f]["text"]))
if odd:
    print("unreadable characters:", dict(odd.most_common(12)))

dropped = work / "dropped"
shutil.rmtree(dropped, ignore_errors=True)
for f, d in decision.items():
    if d != "keep":
        link(src / f, dropped / d / Path(f).name)
save_json(work / "report.json", {f: dict(reason=d, text=texts[f]["text"], align_prob=texts[f]["align_prob"],
                                         speaker=by_file[f]["speaker"], relabeled_to=label_of[f] if f in relabeled else None,
                                         diarization=diar.get(f), sim_own=round(sim_own[f], 3), closer_to=closer.get(f))
                                 for f, d in decision.items() if d != "keep" or f in relabeled})
print(f"dropped chunks linked into {dropped}/<reason>/")

# review page: merged speakers, near misses, relabeled / mixed / wrong-speaker chunks
rev = work / "review"
shutil.rmtree(rev, ignore_errors=True)
shutil.rmtree(work / "speakers", ignore_errors=True)        # the older review folder


def sample(l):
    return max(by_label[l], key=lambda c: min(c["duration"], 12))   # one of its long chunks, about 12 s


def player(c, tag):
    p = f"{tag}_{Path(c['file']).name}"
    link(src / c["file"], rev / p)
    return f"<audio controls preload='none' src='{p}'></audio>"


def votes(f):
    d = diar.get(f)
    return f"systems agreeing with the label: {d['agree']}/{d['votes']}, mixed votes: {d['mixed']}" if d else "no diarization check"


body = ["<h2>Merged into one speaker</h2>"]
for sid, s_ in speakers.items():
    if len(s_["labels"]) > 1:
        body.append(f"<h3>{sid}: {s_['minutes']:.1f} min</h3>" + "".join(
            f"<p>{player(sample(l), sid)} {html.escape(l)}, {minutes[l]:.1f} min</p>" for l in s_["labels"]))
body.append(f"<h2>Not merged, but close (voice similarity {MERGE_SIM - 0.15:.2f}–{MERGE_SIM:.2f})</h2>")
for s_, a, b in near[:N_REVIEW]:
    body.append(f"<p>{s_:.2f}: {player(sample(a), 'near')} {html.escape(a)} · {player(sample(b), 'near')} {html.escape(b)}</p>")
for title, fs in [("Relabeled (kept with the new label)", [f for f in relabeled if decision[f] == "keep"]),
                  ("Mixed speakers (dropped)", [f for f, d in decision.items() if d == "mixed_speakers"]),
                  ("Uncertain speaker (dropped)", [f for f, d in decision.items() if d == "uncertain_speaker"]),
                  ("Doubted by the systems, the voice confirms the label (kept)", sorted(doubted_kept)),
                  (f"The systems agree, but the voice is {WRONG_MARGIN}+ closer to another speaker (kept)",
                   [f for f in closer if decision[f] == "keep" and f not in relabeled and f not in doubted_kept])]:
    body.append(f"<h2>{title}: {len(fs)}</h2>")
    for f in fs[:N_REVIEW]:
        c = by_file[f]
        line = f"<p>{player(c, 'chunk')} {c['duration']:.1f} s · {html.escape(c['text'][:100])}<br>{votes(f)}<br>"
        line += f"labelled {html.escape(c['speaker'])} {player(sample(c['speaker']), 'own')}"
        if f in relabeled:
            line += f" → now {html.escape(label_of[f])} {player(sample(label_of[f]), 'new')}"
        elif f in closer:
            line += f" · voice closer to {html.escape(closer[f][0])} {player(sample(closer[f][0]), 'other')}"
        body.append(line + "</p>")
(rev / "index.html").write_text("<!DOCTYPE html><meta charset='utf-8'><title>Speakers and diarization</title>"
                                "<style>body{font-family:system-ui;margin:24px}p{margin:4px 0 12px}</style>"
                                "<h1>Speakers and diarization</h1><p>Each sample player is one chunk of that speaker "
                                "label, for comparison.</p>" + "".join(body), encoding="utf-8")
print(f"review: {rev / 'index.html'}")

                   chunks   hours
keep                  506    1.10   (of them relabeled: 69; doubted by the systems but the voice confirms the label: 38)
unreadable              0    0.00
bad_alignment          31    0.02
overlap                 2    0.00
mixed_speakers         10    0.03
uncertain_speaker      55    0.04
small_speaker           0    0.00
relabeled: {'002_speaker_2 -> 002_speaker_4': 2, '002_speaker_2 -> 002_speaker_3': 16, '002_speaker_2 -> 002_speaker_5': 3, '002_speaker_2 -> 002_speaker_1': 11, '002_speaker_6 -> 002_speaker_3': 6, '002_speaker_6 -> 002_speaker_1': 4, '002_speaker_6 -> 002_speaker_4': 2, '002_speaker_2 -> 002_speaker_6': 2, '002_speaker_1 -> 002_speaker_5': 1, '002_speaker_1 -> 002_speaker_6': 2, '002_speaker_1 -> 002_speaker_3': 4, '002_speaker_6 -> 002_speaker_2': 2, '002_speaker_6 -> 002_speaker_0': 1, '002_speaker_6 -> 002_speaker_7': 2, '002_speaker_1 -> 002_speaker_4': 1, '002_speaker_4 -> 002_speaker_1': 1, '002_speaker_4 -> 002_speaker_3': 2

## 5. Build `dataset_train/`

In [8]:
keep = [c for c in chunks if decision[c["file"]] == "keep"]
rng = random.Random(SEED)
val = set()
for sid in sorted({new_id[label_of[c["file"]]] for c in keep}):
    cs = sorted(c["file"] for c in keep if new_id[label_of[c["file"]]] == sid and 1.5 <= c["duration"] <= 15)
    n = round(VAL_SHARE * sum(1 for c in keep if new_id[label_of[c["file"]]] == sid))
    val |= set(rng.sample(cs, min(n, len(cs))))


def build(c):
    x, sr = sf.read(src / c["file"], dtype="float32")
    loud = pyloudnorm.Meter(sr).integrated_loudness(x)
    gain = TARGET_LUFS - loud if np.isfinite(loud) else 0.0
    peak = float(np.abs(x).max()) or 1e-9
    gain = min(gain, PEAK_DBFS - 20 * np.log10(peak))        # never above PEAK_DBFS
    y = x * np.float32(10 ** (gain / 20))
    assert sr == OUT_SR, (c["file"], sr)                     # never resampled
    y = np.clip(y, -1.0, 1.0)
    sf.write(tmp / c["file"], y, OUT_SR, subtype="PCM_16")
    return c["file"], round(len(y) / OUT_SR, 3), round(float(gain), 1), np.isfinite(loud) and gain < TARGET_LUFS - loud - 0.05


tmp = out_dir.with_name(out_dir.name + ".tmp")
shutil.rmtree(tmp, ignore_errors=True)
(tmp / "chunks").mkdir(parents=True)
threads = torch.get_num_threads()
torch.set_num_threads(1)                                     # one per worker
try:
    with ThreadPoolExecutor(WORKERS) as pool:
        built = {f: (d, g, lim) for f, d, g, lim in tqdm(pool.map(build, keep), total=len(keep), desc="build")}
finally:
    torch.set_num_threads(threads)

rows = []
for c in keep:
    f, t = c["file"], texts[c["file"]]
    d, g, _ = built[f]
    dg = diar.get(f)
    rows.append(dict(file=f, speaker=new_id[label_of[f]], text=t["text"], duration=d,
                     split="val" if f in val else "train", speaker_label=label_of[f],
                     **({"speaker_label_before": relabeled[f]} if f in relabeled else {}),
                     diarization=f"{dg['agree']}/{dg['votes']}" if dg else None,
                     **({"text_original": t["text_in"]} if t["text"] != t["text_in"] else {}),
                     align_prob=t["align_prob"], overlap_sec=dg["overlap_sec"] if dg else None, gain_db=g))
kept_speakers = {s: dict(speakers[s], minutes=round(sum(r["duration"] for r in rows if r["speaker"] == s) / 60, 1),
                         chunks=sum(1 for r in rows if r["speaker"] == s)) for s in sorted({r["speaker"] for r in rows})}
save_json(tmp / "metadata.json", dict(
    sample_rate=OUT_SR, podcasts=meta["podcasts"], speakers=kept_speakers,
    settings=dict(MIN_EMB_SEC=MIN_EMB_SEC, MERGE_SIM=MERGE_SIM, MIN_ALIGN_PROB=MIN_ALIGN_PROB,
                  MAX_OVERLAP_SEC=MAX_OVERLAP_SEC, RELABEL=RELABEL, RELABEL_MARGIN=RELABEL_MARGIN, MIN_OTHER_SEC=MIN_OTHER_SEC,
                  MIN_VOTES=need, DIARIZATION=REF, systems=voters,
                  MIN_SPEAKER_MIN=MIN_SPEAKER_MIN, TARGET_LUFS=TARGET_LUFS, PEAK_DBFS=PEAK_DBFS,
                  VAL_SHARE=VAL_SHARE, SEED=SEED, source=str(src)),
    chunks=rows), indent=2)
for split in ("train", "val"):
    (tmp / f"{split}.txt").write_text("".join(f"{r['file']}|{r['text'].replace('|', ' ')}|{r['speaker']}\n"
                                              for r in rows if r["split"] == split), encoding="utf-8")
if out_dir.exists():
    shutil.rmtree(out_dir)                                  # the previous build
os.replace(tmp, out_dir)
n_lim = sum(1 for v in built.values() if v[2])
print(f"{out_dir.name}: {len(rows)} chunks, {sum(r['duration'] for r in rows) / 3600:.2f} h, {OUT_SR} Hz, "
      f"{len(kept_speakers)} speakers; train {sum(r['split'] == 'train' for r in rows)}, val {len(val)}")
print(f"gain applied, percentiles 5/50/95: {np.round(np.percentile([r['gain_db'] for r in rows], [5, 50, 95]), 1)} dB; "
      f"{n_lim} chunks stayed below {TARGET_LUFS} LUFS so that no sample goes above {PEAK_DBFS} dBFS")

# check
m2 = json.load(open(out_dir / "metadata.json", encoding="utf-8"))
wavs = {p.name for p in (out_dir / "chunks").glob("*.wav")}
names = {Path(r["file"]).name for r in m2["chunks"]}
bad = [r["file"] for r in m2["chunks"] if abs(sf.info(out_dir / r["file"]).frames / OUT_SR - r["duration"]) > 0.002]
lines = sum(1 for s in ("train", "val") for _ in open(out_dir / f"{s}.txt", encoding="utf-8"))
print("every chunk has its wav:", names == wavs and len(names) == len(m2["chunks"]),
      "| durations match:", "all" if not bad else f"NO: {bad[:5]}",
      "| manifest lines:", lines, "| unreadable texts:", sum(bool(UNREADABLE.search(r["text"])) for r in m2["chunks"]),
      "| sample rates:", dict(Counter(sf.info(out_dir / r["file"]).samplerate for r in m2["chunks"][:200])))
lufs = [pyloudnorm.Meter(OUT_SR).integrated_loudness(sf.read(out_dir / r["file"], dtype="float32")[0])
        for r in random.Random(1).sample(m2["chunks"], min(100, len(m2["chunks"])))]
print("loudness of 100 chunks now, percentiles 5/50/95:", np.round(np.percentile(lufs, [5, 50, 95]), 1), "LUFS")
print(f"{'speaker':8} {'minutes':>8} {'chunks':>7}  labels")
for s, v in m2["speakers"].items():
    print(f"{s:8} {v['minutes']:8.1f} {v['chunks']:7d}  {', '.join(v['labels'])}")

dataset_train: 506 chunks, 1.10 h, 48000 Hz, 9 speakers; train 495, val 11
gain applied, percentiles 5/50/95: [-5.2 -0.1  8.4] dB; 75 chunks stayed below -23.0 LUFS so that no sample goes above -1.0 dBFS
every chunk has its wav: True | durations match: all | manifest lines: 506 | unreadable texts: 0 | sample rates: {48000: 200}
loudness of 100 chunks now, percentiles 5/50/95: [-24.7 -23.  -23. ] LUFS
speaker   minutes  chunks  labels
spk000       42.1     146  001_speaker_0
spk001        6.7      91  002_speaker_1
spk002        4.4      55  002_speaker_6
spk003        3.6      55  002_speaker_2
spk004        4.5      76  002_speaker_3
spk005        2.1      33  002_speaker_5
spk006        1.5      32  002_speaker_7
spk007        0.8      15  002_speaker_4
spk008        0.3       3  002_speaker_0
